In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement <strong>Sliding Window Self-Attention</strong> for a given set of matrices.
  Before introducing the sliding window version, let's first recall standard Self-Attention.
</p>

<h3>1. Standard Softmax Attention</h3>
<p>
  Given query matrix <code>Q</code>, key matrix <code>K</code>, and value matrix <code>V</code>, each position <code>i</code> attends to all positions <code>j</code> using a softmax-weighted sum:
</p>

<p style="text-align:center;">
  $ \text{score}_{i,j} = \frac{Q_i \cdot K_j}{\sqrt{d}} $
</p>

<p style="text-align:center;">
  $ \text{output}_i = \sum_{j=1}^{M} \text{softmax}(\text{score}_{i,*})_j \cdot V_j $
</p>

<p>
  In other words, each query computes similarity with all keys, applies a softmax to get attention weights, and then computes a weighted sum of values.
</p>

<h3>2. Sliding Window Self-Attention</h3>
<p>
  Sliding Window Attention modifies standard attention by restricting each query to attend only to a local window around its position.
</p>

<ul>
  <li>For each position <code>i</code>, only consider the keys and values within a window of size <code>window_size</code> around <code>i</code> (positions <code>[i-window_size, ..., i+window_size]</code>).</li>
  <li>Compute similarity scores between <code>Q<sub>i</sub></code> and the keys in this window:</li>
</ul>

<p style="text-align:center;">
  $ \text{score}_{i,j} = \frac{Q_i \cdot K_j}{\sqrt{d}} $
</p>

<ul>
  <li>Apply <code>softmax</code> over these local scores to obtain attention weights.</li>
  <li>Use the weights to compute a weighted average of the values in the same window:</li>
</ul>

<p style="text-align:center;">
  $ \text{output}_i = \sum_{j \in [i-\text{window_size}, \, i+\text{window_size}]} \text{softmax}(\text{score}_{i,*})_j \cdot V_j $
</p>

<p>
  In short, each query only attends to its nearby neighbors.
</p>

<svg width="320" height="320" viewBox="0 0 320 320" xmlns="http://www.w3.org/2000/svg"
     style="display:block; margin:20px auto;">
  <style>
    text { font-family: monospace; fill: #cccccc; }
  </style>
  <!-- Background -->
  <rect width="320" height="320" fill="#1a1a1a" rx="6"/>

  <!-- Axis labels -->
  <text x="185" y="16" text-anchor="middle" font-size="11">key position &#x2192;</text>
  <text x="14" y="168" text-anchor="middle" font-size="11" transform="rotate(-90,14,168)">query position</text>

  <!-- Column headers (positions 0-7) -->
  <g font-size="10" text-anchor="middle">
    <text x="62" y="36">0</text>
    <text x="93" y="36">1</text>
    <text x="124" y="36">2</text>
    <text x="155" y="36">3</text>
    <text x="186" y="36">4</text>
    <text x="217" y="36">5</text>
    <text x="248" y="36">6</text>
    <text x="279" y="36">7</text>
  </g>

  <!-- Row headers (positions 0-7) -->
  <g font-size="10" text-anchor="middle">
    <text x="34" y="58">0</text>
    <text x="34" y="89">1</text>
    <text x="34" y="120">2</text>
    <text x="34" y="151">3</text>
    <text x="34" y="182">4</text>
    <text x="34" y="213">5</text>
    <text x="34" y="244">6</text>
    <text x="34" y="275">7</text>
  </g>

  <!-- 8x8 grid: cell is blue (#2a4a7a) if |row-col| <= 2, else dark (#2a2a2a) -->
  <!-- Row 0: cols 0,1,2 blue -->
  <rect x="46" y="42" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="77" y="42" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="108" y="42" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="139" y="42" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="170" y="42" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="201" y="42" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="232" y="42" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="263" y="42" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <!-- Row 1: cols 0,1,2,3 blue -->
  <rect x="46" y="73" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="77" y="73" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="108" y="73" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="139" y="73" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="170" y="73" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="201" y="73" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="232" y="73" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="263" y="73" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <!-- Row 2: cols 0,1,2,3,4 blue -->
  <rect x="46" y="104" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="77" y="104" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="108" y="104" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="139" y="104" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="170" y="104" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="201" y="104" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="232" y="104" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="263" y="104" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <!-- Row 3: cols 1,2,3,4,5 blue -->
  <rect x="46" y="135" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="77" y="135" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="108" y="135" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="139" y="135" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="170" y="135" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="201" y="135" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="232" y="135" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="263" y="135" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <!-- Row 4: cols 2,3,4,5,6 blue -->
  <rect x="46" y="166" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="77" y="166" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="108" y="166" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="139" y="166" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="170" y="166" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="201" y="166" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="232" y="166" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="263" y="166" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <!-- Row 5: cols 3,4,5,6,7 blue -->
  <rect x="46" y="197" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="77" y="197" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="108" y="197" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="139" y="197" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="170" y="197" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="201" y="197" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="232" y="197" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="263" y="197" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <!-- Row 6: cols 4,5,6,7 blue -->
  <rect x="46" y="228" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="77" y="228" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="108" y="228" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="139" y="228" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="170" y="228" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="201" y="228" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="232" y="228" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="263" y="228" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <!-- Row 7: cols 5,6,7 blue -->
  <rect x="46" y="259" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="77" y="259" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="108" y="259" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="139" y="259" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="170" y="259" width="31" height="31" fill="#2a2a2a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="201" y="259" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="232" y="259" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>
  <rect x="263" y="259" width="31" height="31" fill="#2a4a7a" stroke="#1a1a1a" stroke-width="1"/>

  <!-- Grid border -->
  <rect x="46" y="42" width="248" height="248" fill="none" stroke="#555555" stroke-width="1"/>

  <!-- Label: window_size = 2 -->
  <text x="170" y="308" text-anchor="middle" font-size="12" fill="#999999">window_size = 2</text>
</svg>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The
    <code>solve</code> function signature must remain unchanged
  </li>
  <li>The final result must be stored in the output matrix
    <code>output</code>
  </li>
</ul>
<h2>Example 1:</h2>
<p>
<strong>Input:</strong><br>
<code>Q</code> (2×4):
$$
\begin{bmatrix}
1.0 & 0.0 & 0.0 & 0.0 \\
0.0 & 1.0 & 0.0 & 0.0
\end{bmatrix}
$$
<code>K</code> (2×4):
$$
\begin{bmatrix}
1.0 & 0.0 & 0.0 & 0.0 \\
0.0 & 1.0 & 0.0 & 0.0
\end{bmatrix}
$$
<code>V</code> (2×4):
$$
\begin{bmatrix}
1.0 & 2.0 & 3.0 & 4.0 \\
5.0 & 6.0 & 7.0 & 8.0
\end{bmatrix}
$$
<code>window_size</code>: 1
</p>

<p>
<strong>Output:</strong><br>
<code>output</code> (2×4):
$$
\begin{bmatrix}
2.5101628 & 3.5101628 & 4.510163 & 5.510163 \\
3.4898374 & 4.4898376 & 5.4898376 & 6.489837
\end{bmatrix}
$$
</p>


<h2>Example 2:</h2>
<p>
  <strong>Input:</strong><br>
  <code>Q</code> (2×3):
  $$
  \begin{bmatrix}
  0.0 & 0.0 & 0.0 \\
  0.0 & 1.0 & 0.0
  \end{bmatrix}
  $$
  <code>K</code> (2×3):
  $$
  \begin{bmatrix}
  1.0 & 0.0 & 0.0 \\
  0.0 & 1.0 & 0.0
  \end{bmatrix}
  $$
  <code>V</code> (2×3):
  $$
  \begin{bmatrix}
  1.0 & 2.0 & 3.0 \\
  5.0 & 6.0 & 7.0
  \end{bmatrix}
  $$
  <code>window_size</code>: 1
  </p>

  <p>
  <strong>Output:</strong><br>
  <code>output</code> (2×3):
  $$
  \begin{bmatrix}
  3.0 & 4.0 & 5.0 \\
  3.5618298 & 4.56183 & 5.5618296
  \end{bmatrix}
  $$
  </p>



<h2>Constraints</h2>
<ul>
  <li>Matrix <code>Q</code>, <code>K</code>, and <code>V</code> are all of size <code>M×d</code></li>
  <li>1 &le; <code>M</code> &le; 10000</li>
  <li>1 &le; <code>d</code> &le; 128</li>
  <li>1 &le; <code>window_size</code> &le; 32</li>
  <li>All elements in <code>Q</code>, <code>K</code>, and <code>V</code> are sampled from<code>[-100.0, 100.0]</code></li>
  <li>Data type for all matrices is <code>float32</code></li>

  <li>Performance is measured with <code>M</code> = 5,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// Q, K, V, output are device pointers
extern "C" void solve(const float* Q, const float* K, const float* V, float* output, int M, int d,
                      int window_size) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# Q, K, V, output are tensors on the GPU
@cute.jit
def solve(
    Q: cute.Tensor,
    K: cute.Tensor,
    V: cute.Tensor,
    output: cute.Tensor,
    M: int,
    d: int,
    window_size: int,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# Q, K, V are tensors on the GPU
@jax.jit
def solve(Q: jax.Array, K: jax.Array, V: jax.Array, M: int, d: int, window_size: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# Q, K, V, output are device pointers (i.e. pointers to memory on the GPU)
@export
def solve(
    Q: UnsafePointer[Float32, MutExternalOrigin],
    K: UnsafePointer[Float32, MutExternalOrigin],
    V: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    M: Int32,
    d: Int32,
    window_size: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# Q, K, V, output are tensors on the GPU
def solve(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    output: torch.Tensor,
    M: int,
    d: int,
    window_size: int,
):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# Q, K, V, output are tensors on the GPU
def solve(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    output: torch.Tensor,
    M: int,
    d: int,
    window_size: int,
):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="Sliding Window Self-Attention",
            atol=1e-05,
            rtol=1e-05,
            num_gpus=1,
            access_tier="free",
        )

    def reference_impl(
        self,
        Q: torch.Tensor,
        K: torch.Tensor,
        V: torch.Tensor,
        output: torch.Tensor,
        M: int,
        d: int,
        window_size: int,
    ):
        assert Q.shape == K.shape == V.shape == output.shape == (M, d)

        scores = (Q @ K.T) / (d**0.5)

        idxs = torch.arange(M)
        mask = (idxs[None, :] - idxs[:, None]).abs() > window_size
        mask = mask.to(Q.device)
        scores.masked_fill_(mask, float("-inf"))
        attn = torch.softmax(scores, dim=1)

        torch.matmul(attn, V, out=output)

    def get_solve_signature(self) -> Dict[str, Any]:
        return {
            "Q": (ctypes.POINTER(ctypes.c_float), "in"),
            "K": (ctypes.POINTER(ctypes.c_float), "in"),
            "V": (ctypes.POINTER(ctypes.c_float), "in"),
            "output": (ctypes.POINTER(ctypes.c_float), "out"),
            "M": (ctypes.c_int, "in"),
            "d": (ctypes.c_int, "in"),
            "window_size": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        Q = torch.tensor([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]], device="cuda", dtype=dtype)
        K = torch.tensor([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]], device="cuda", dtype=dtype)
        V = torch.tensor([[1.0, 2.0, 3.0, 4.0], [5.0, 6.0, 7.0, 8.0]], device="cuda", dtype=dtype)
        output = torch.empty(2, 4, device="cuda", dtype=dtype)
        return {"Q": Q, "K": K, "V": V, "output": output, "M": 2, "d": 4, "window_size": 1}

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float32
        tests = []

        # basic_example
        tests.append(
            {
                "Q": torch.tensor(
                    [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]], device="cuda", dtype=dtype
                ),
                "K": torch.tensor(
                    [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]], device="cuda", dtype=dtype
                ),
                "V": torch.tensor(
                    [[1.0, 2.0, 3.0, 4.0], [5.0, 6.0, 7.0, 8.0]], device="cuda", dtype=dtype
                ),
                "output": torch.empty(2, 4, device="cuda", dtype=dtype),
                "M": 2,
                "d": 4,
                "window_size": 1,
            }
        )

        # basic_example
        tests.append(
            {
                "Q": torch.tensor([[0.0, 0.0, 0.0], [0.0, 1.0, 0.0]], device="cuda", dtype=dtype),
                "K": torch.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]], device="cuda", dtype=dtype),
                "V": torch.tensor([[1.0, 2.0, 3.0], [5.0, 6.0, 7.0]], device="cuda", dtype=dtype),
                "output": torch.empty(2, 3, device="cuda", dtype=dtype),
                "M": 2,
                "d": 3,
                "window_size": 1,
            }
        )

        # zero_matrices
        tests.append(
            {
                "Q": torch.zeros((3, 5), device="cuda", dtype=dtype),
                "K": torch.zeros((3, 5), device="cuda", dtype=dtype),
                "V": torch.zeros((3, 5), device="cuda", dtype=dtype),
                "output": torch.empty(3, 5, device="cuda", dtype=dtype),
                "M": 3,
                "d": 5,
                "window_size": 2,
            }
        )

        # mixed_values
        tests.append(
            {
                "Q": torch.tensor(
                    [[-1.0, 2.0, -3.0], [4.0, -5.0, 6.0], [-7.0, 8.0, -9.0], [10.0, -11.0, 12.0]],
                    device="cuda",
                    dtype=dtype,
                ),
                "K": torch.tensor(
                    [[2.0, -1.0, 3.0], [-4.0, 5.0, -6.0], [7.0, -8.0, 9.0], [-10.0, 11.0, -12.0]],
                    device="cuda",
                    dtype=dtype,
                ),
                "V": torch.tensor(
                    [[1.0, 0.5, -0.5], [-1.0, 2.0, 3.0], [4.0, -2.0, 1.0], [0.0, 1.0, -1.0]],
                    device="cuda",
                    dtype=dtype,
                ),
                "output": torch.empty(4, 3, device="cuda", dtype=dtype),
                "M": 4,
                "d": 3,
                "window_size": 2,
            }
        )

        # large_matrices
        tests.append(
            {
                "Q": torch.empty((128, 32), device="cuda", dtype=dtype).uniform_(-0.1, 0.1),
                "K": torch.empty((128, 32), device="cuda", dtype=dtype).uniform_(-0.1, 0.1),
                "V": torch.empty((128, 32), device="cuda", dtype=dtype).uniform_(-0.1, 0.1),
                "output": torch.empty(128, 32, device="cuda", dtype=dtype),
                "M": 128,
                "d": 32,
                "window_size": 8,
            }
        )

        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        M, d, window_size = 5000, 64, 16
        Q = torch.empty((M, d), device="cuda", dtype=dtype).uniform_(-100, 100)
        K = torch.empty((M, d), device="cuda", dtype=dtype).uniform_(-100, 100)
        V = torch.empty((M, d), device="cuda", dtype=dtype).uniform_(-100, 100)
        output = torch.empty(M, d, device="cuda", dtype=dtype)
        return {
            "Q": Q,
            "K": K,
            "V": V,
            "output": output,
            "M": M,
            "d": d,
            "window_size": window_size,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
